# Name Hierarchy Levels — SynBio Papers (Mid & High)

Assigns globally unique, publication-ready names to the mid-level and high-level
topic groups for **SynBio Papers**. For each group the LLM sees the low-level
sub-topics (name + description) it contains, then returns one name per group via
OpenAI function calling. Prompts come from `prompts_hierarchy.yaml`.

> Run `get_topic_hierarchy.ipynb` (this folder) **first**.

**Updates** `assets/reports/papers_topic_name_hierarchy.tsv` with `mid_name`
and `high_name` columns.

In [ ]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 04-topic_hierarchy/, where the aux/
# package resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from aux.paths import REPORTS_DIR, OPENAI_MODEL, set_seed
from aux.naming import (
    load_prompts, make_client, build_system_prompt,
    name_hierarchy_level, load_naming_inputs, save_named_hierarchy,
)

set_seed()

# ── CONFIG: SynBio Papers ──────────────────────────────────────────────────
PREFIX = "papers"

prompts = load_prompts()
client = make_client()
system_prompt = build_system_prompt(prompts)

topic_names, hierarchy = load_naming_inputs(PREFIX)
print(f"{PREFIX}: {len(topic_names)} low-level topics | "
      f"mid groups {hierarchy[hierarchy['mid'] >= 0]['mid'].nunique()} | "
      f"high groups {hierarchy[hierarchy['high'] >= 0]['high'].nunique()}")

## 1. Name the mid- and high-level groups

In [ ]:
mid_names = name_hierarchy_level(
    hierarchy, topic_names, level_col="mid", label="mid",
    client=client, system_prompt=system_prompt, model=OPENAI_MODEL,
)
high_names = name_hierarchy_level(
    hierarchy, topic_names, level_col="high", label="high",
    client=client, system_prompt=system_prompt, model=OPENAI_MODEL,
)

## 2. Add the name columns and save

In [ ]:
hierarchy["mid_name"] = hierarchy["mid"].map(mid_names)
hierarchy["high_name"] = hierarchy["high"].map(high_names)
save_named_hierarchy(hierarchy, PREFIX)

print(f"Saved → {REPORTS_DIR / f'{PREFIX}_topic_name_hierarchy.tsv'}")
hierarchy.head(10)

## 3. Summary

In [ ]:
n_mid = hierarchy["mid_name"].notna().sum()
n_high = hierarchy["high_name"].notna().sum()
print(f"{PREFIX}: {n_mid} topics with mid_name ({hierarchy['mid_name'].dropna().nunique()} unique), "
      f"{n_high} with high_name ({hierarchy['high_name'].dropna().nunique()} unique)")